In [2]:
#!pip install opencv-python
import os
os.environ["QT_QPA_PLATFORM"] = "xcb"
#Gestion de l'affichage pour opencv wailand

In [3]:
import cv2 
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
racine_entrainement = "/home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/CCPD_Dataset_preproc/train/"
save_dir = "/home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models_TP1"

In [ ]:
def densite_zones(img, line_segmentation, column_segmentation):
    """
    Calcule le vecteur de densité de pixels pour une image binaire
    découpée en (line_segmentation x column_segmentation) zones.
    """

    caracteristic_vector = []

    for bande in np.array_split(img, line_segmentation, axis=0):
        for zone in np.array_split(bande, column_segmentation, axis=1):
            densite = np.count_nonzero(zone) / zone.size
            caracteristic_vector.append(densite)

    caracteristic_vector = np.array(caracteristic_vector)

    return caracteristic_vector if caracteristic_vector.size > 0 else np.nan

In [ ]:
line_segmentation = 5
column_segmentation = 5

# repertoire pour sauvegarder les profils
os.makedirs(save_dir, exist_ok=True)

for chemin_dossier, sous_dossiers, fichiers in os.walk(racine_entrainement):
    classe = os.path.basename(chemin_dossier)
    resultats = []

    for fichier in fichiers:
        if fichier.lower().endswith((".png")):
            chemin_complet = os.path.join(chemin_dossier, fichier)
            img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            gauche = moyenne_distance_bord(img, number_lines=number_lines, cote="gauche")
            droite = moyenne_distance_bord(img, number_lines=number_lines, cote="droite")

# Sens de la longueur du profil axis = 0
            profil = np.concatenate((gauche,droite),axis=0)

            resultats.append(profil)
            # Matrice de dimention (nombre d'images, nombre de lignes * 2) ou (nb_images, d) donc la moyenne par collonne
            
    means_resultats  = np.mean(resultats, axis=0)

    if classe != "" :
        # fabrique le chemin du fichier dans le dossier central
        name_file = os.path.join(save_dir, classe)
        np.save(name_file, means_resultats)
        print(f"Fichier sauvegardé : {name_file}")


/home/julien/.pyenv/versions/archlinux/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/julien/.pyenv/versions/archlinux/lib/python3.11/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/0
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/1
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/2
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/3
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/4
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/5
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/6
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/7
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/8
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/9
Fichier sauvegardé : /home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Tranning_Models/A

In [61]:
def distance_(img,model, number_lines=5):
    gauche = moyenne_distance_bord(img, number_lines=number_lines, cote="gauche")
    droite = moyenne_distance_bord(img, number_lines=number_lines, cote="droite")
    profil = np.concatenate((gauche,droite),axis=0)
    return np.linalg.norm(profil - model)

def predict(img,models, number_lines=5):
    """
    Calcule la prédiction et les probabilités de chaque classe
    pour une image donnée.

    En fait le truc c'est que on calcule la distance euclidienne pour chaque modèle et on fait un distro de proba : 
    """
    # Calculer toutes les distances
    distances = {}
    for classe, model in models.items():
        distances[classe] = distance_(img, model, number_lines=number_lines)

    # Conversion en probabilités 
    dist_values = np.array(list(distances.values()))
    exp_neg = np.exp(-dist_values)
    proba_values = exp_neg / np.sum(exp_neg)

    # Associer probas classes
    probabilites = dict(zip(distances.keys(), proba_values))

    # Trouver la classe la plus probable 
    prediction = max(probabilites, key=probabilites.get)

    return prediction, probabilites


def load_models(save_dir):
    models = {}
    for fichier in os.listdir(save_dir):
        if fichier.endswith(".npy"):
            classe = os.path.splitext(fichier)[0]
            chemin_complet = os.path.join(save_dir, fichier)
            model = np.load(chemin_complet)
            models[classe] = model
    return models
        

In [62]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Chargement du modèle
models = load_models(save_dir)
number_lines = 5

# Dossiers
racine_test = "/home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/CCPD_Dataset_preproc/test/"
save_dir_proba = "/home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP1/Training_Test"
os.makedirs(save_dir_proba, exist_ok=True)

# Pour l’évaluation
y_true, y_pred = [], []

# Parcours des images de test
for chemin_dossier, _, fichiers in os.walk(racine_test):
    classe_reelle = os.path.basename(chemin_dossier)
    if not classe_reelle or classe_reelle == os.path.basename(racine_test):
        continue

    for fichier in fichiers:
        if not fichier.lower().endswith(".png"):
            continue
        chemin_complet = os.path.join(chemin_dossier, fichier)
        img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # --- Prédiction ---
        prediction, probabilites = predict(img, models, number_lines=number_lines)

        # --- Stockage pour évaluation ---
        y_true.append(classe_reelle)
        y_pred.append(prediction)

        # --- Sauvegarde du vecteur de probas ---
        vec_probabilites = np.array(list(probabilites.values()))
        nom_sans_ext = os.path.splitext(fichier)[0]
        name_file = os.path.join(save_dir_proba, nom_sans_ext + "_proba.npy")
        np.save(name_file, vec_probabilites)

# --- Évaluation globale ---
acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=sorted(models.keys()))

print("\n=== Résultats d'évaluation ===")
print(f"Nombre total d'images testées : {len(y_true)}")
print(f"Taux global de reconnaissance : {acc:.2%}")

# --- Tableau synthétique par classe ---
print("\n=== Rapport par classe ===")
report = classification_report(y_true, y_pred, digits=3)
print(report)



=== Résultats d'évaluation ===
Nombre total d'images testées : 475
Taux global de reconnaissance : 81.47%

=== Rapport par classe ===
              precision    recall  f1-score   support

           0      0.806     0.926     0.862        27
           1      0.833     0.312     0.455        32
           2      0.864     0.679     0.760        28
           3      1.000     0.889     0.941        36
           4      1.000     1.000     1.000        18
           5      1.000     0.969     0.984        32
           6      0.957     1.000     0.978        44
           7      0.879     0.967     0.921        30
           8      0.918     0.957     0.938        47
           9      1.000     1.000     1.000        32
           A      0.571     0.800     0.667         5
           B      0.333     0.600     0.429         5
           C      1.000     1.000     1.000         6
           D      0.364     0.800     0.500         5
           E      1.000     1.000     1.000         4
